# 4. Reaction DFT, TS, and IRC

Input: a reaction CSV and the reactant result table. The first stage can stop after reaction free energies; the later stages perform conformer, TS, IRC, solvent, and distortion/interaction work. These calculations are expensive and contain historical alternative routes.

Saved outputs were cleared in this split. The archived source notebook retains the historical outputs and execution counts. Set the job paths before running any calculation.


In [ ]:
from pathlib import Path
import os
import sys

# Resolve the repository root from the kernel working directory, or set CYCLOADD_REPO_ROOT.
repo_root_env = os.environ.get("CYCLOADD_REPO_ROOT")
if repo_root_env:
    REPO_ROOT = Path(repo_root_env).expanduser().resolve()
    if not (REPO_ROOT / "Code").is_dir() or not (REPO_ROOT / "Data").is_dir():
        raise RuntimeError(f"CYCLOADD_REPO_ROOT is not a repository root: {REPO_ROOT}")
else:
    REPO_ROOT = next(
        (parent for parent in (Path.cwd(), *Path.cwd().parents)
         if (parent / "Code").is_dir() and (parent / "Data").is_dir()),
        None,
    )
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter in this repository or set CYCLOADD_REPO_ROOT")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import shutil
from Code import main, xtb_process

target_dir = os.environ.get("CYCLOADD_REACTION_WORK_DIR", r"E:\work\Secondary_Selection\ZINC_1")
smiles_dir = os.environ.get("CYCLOADD_SMILES_DIR", r"E:\work\Secondary_Selection\Diene_Ene_Smiles")
smiles_csv_dir = os.path.join(smiles_dir, "all_smiles.csv")
reaction_file_dir = os.environ.get("CYCLOADD_REACTION_FILE", os.path.join(target_dir, "ZINC_1.csv"))
Result_csv = str(REPO_ROOT / "output" / "reaction_dft_result.csv")
Path(Result_csv).parent.mkdir(parents=True, exist_ok=True)
acquire_title = False


# 4. DFT Calculation Reactions

## Configure one reaction job

Set the input and output for this calculation. The exported CSVs default to new files in `output/` so a trial run does not overwrite saved research tables.


In [ ]:
if not os.path.isdir(target_dir):
    os.mkdir(target_dir)
run_sugan_file = str(REPO_ROOT / 'Data' / 'GaussianSample' / 'run_new.sh')
new_sugan_file = target_dir + "/run_new.sh"
shutil.copy(run_sugan_file, new_sugan_file)

In [ ]:
main.collect_reaction_file(reaction_file_dir=reaction_file_dir, target_dir=target_dir, smiles_dir=smiles_dir, smiles_csv_dir=smiles_csv_dir, n_cores=60, acquire_title=acquire_title, check_charge=0, max_attemp_num=50, acquire_map=True)

In [ ]:
xtb_process.shift_to_parra(target_dir + "/first_xtb")

In [ ]:
main.smiles_DFT_calc(target_dir, has_title=1)

In [ ]:
main.error_improve(target_dir, "mol_dft")

In [ ]:
main.SPE_DFT_calc(target_dir, "mol_dft", "mol_dft_eng")

In [ ]:
main.error_improve(target_dir, "mol_dft_eng")

In [ ]:
main.reaction_result_analysis(target_dir, smiles_csv_dir=smiles_csv_dir)

In [ ]:
# if Just Calculate deltaG, Run This and Ignore the Code After
target_csv = target_dir + "/Result.csv"
Rxn_Result_csv = str(REPO_ROOT / 'output' / 'reaction_deltaG_result.csv')
shutil.copy(target_csv, Rxn_Result_csv)

### Detail Conformer Search

The detailed conformer-search cells below and the direct Gaussian route are alternative historical procedures. Select the route appropriate to this reaction job; do not run both merely because they appear in sequence.


In [ ]:
main.reaction_calc_om(target_dir, SELECT_BY_ENG=1, target_cos=-0.5, Difreeze=1, opt_name="mol_dft", om_name='xtb_om', use_gauss=False)

In [ ]:
xtb_process.shift_to_parra(target_dir + "/xtb_om")

In [ ]:
main.smiles_DFT_calc(target_dir, has_title=1, root_dir='xtb_om', dft_dir='om_3', 
                     method='opt=modredundant freq b3lyp/6-31g(d) em=gd3bj', conf_limit=5, rmsd_limit=1, freeze=1)

### Normal Process

In [ ]:
main.reaction_calc_om(target_dir, SELECT_BY_ENG=1, target_cos=-0.5, Difreeze=0, opt_name="mol_dft", om_name='om_3', use_gauss=True)

In [ ]:
xtb_process.shift_to_sugan(target_dir + "/xtb_om", 2)

In [ ]:
main.reaction_calc_om(target_dir, SELECT_BY_ENG=1, target_cos=-0.9, Difreeze=1, opt_name="mol_dft", om_name='xtb_om', use_gauss=False)
# xtb_process.shift_to_sugan(target_dir + "/xtb_om", 1)

In [ ]:
main.reaction_calc_om_2(target_dir,
                        target_cos=-0.5, 
                        Difreeze=1, 
                        gaussian_input=False, 
                        use_gauss=False, 
                        om_name1='xtb_om',
                        om_name2='xtb_om_2')
# xtb_process.shift_to_sugan(target_dir + "/xtb_om_2", 1)

In [ ]:
main.reaction_calc_om_2(target_dir,
                        target_cos=-0.5, 
                        Difreeze=0, 
                        gaussian_input=False, 
                        use_gauss=True, 
                        om_name1='xtb_om_2',
                        om_name2='om_3',
                        )

In [ ]:
main.error_improve(target_dir, "om_3")

In [ ]:
main.ts_Z_E_check(target_dir, ts_name="om_3")

In [ ]:
main.reaction_calc_ts(target_dir, om_name="om_3", )

In [ ]:
main.error_improve(target_dir, "ts")

In [ ]:
main.reaction_calc_irc(target_dir)

In [ ]:
main.error_improve(target_dir, "irc", 'irc_wrong')

In [ ]:
main.reaction_irc_select(target_dir, ts_name='ts')

In [ ]:
main.ts_Z_E_check(target_dir, ts_name="ts")

In [ ]:
main.ts_SPE_DFT_calc(target_dir, 'ts')

In [ ]:
main.error_improve(target_dir, "ts_eng")

In [ ]:
main.ts_SPE_DFT_calc(target_dir, eng_name = "ts_eng_solvent", method="b3lyp/6-311+g(d,p) em=gd3bj scrf=(smd, solvent=water)")

In [ ]:
main.error_improve(target_dir, "ts_eng_solvent")

In [ ]:
main.check_product_stereo(target_dir)

In [ ]:
main.reaction_calc_seperate_reactants(target_dir, type='both')

In [ ]:
main.error_improve(target_dir, 'sep_react')

## Final TS analysis

The separate `pre_diene_reaction` repair case is in part 5. It must not replace this job's `target_dir` before the final analysis.


In [ ]:
target_dict = main.reaction_ts_result_analysis(target_dir, smiles_csv_dir=smiles_csv_dir, write_title=0, ts_eng_name='ts_eng', om_name='om_3')
target_dict

In [ ]:
target_csv = target_dir + "/Result_.csv"
shutil.copy(target_csv, Result_csv)